In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io as sio
import harmonypy as hm
import os

In [ ]:
#print the current scanpy version
print(sc.__version__)

1.9.3


In [ ]:

#filter expression matrices to only include HVGs shared across all datasets
def hvg_selection_and_pooling(adata_matrices, n_top_genes = 1000, mode='intersection'):
    #input n expression matrices paths, output n expression matrices with only the union of the HVGs

    #read adata and find hvgs
    if mode not in ['intersection', 'union']:
        raise ValueError("mode must be either 'intersection' or 'union'")
    if mode == 'intersection':
        # load all the adatas and append genes to list
        hvg_bools = []
        for adata in adata_matrices:
            # CHECK: use either genes or ensemblid should be stored as var index
            hvg_bools.append(adata.var.index)
        # find intersection of hvgs
        hvg_selected =  hvg_bools[0]
        for i in range(1, len(hvg_bools)):
            print(len(hvg_selected), len(hvg_bools[i]))
            hvg_selected = list(set(hvg_selected) & set(hvg_bools[i]))
        print("Number of HVGs: ", len(hvg_selected))
    else:
        hvg_bools = []
        for adata in adata_matrices:

            adata_new = sc.AnnData(X=adata.X.toarray().copy(), dtype=adata.X.dtype)
            print("Processing matrix with shape: ", adata_new.X.shape)
            # check if counts are raw integers
            if not np.all(np.mod(adata_new.X, 1) == 0):
                print("Warning: Expression matrix does not contain raw counts.")

                # Preprocess the data
                sc.pp.normalize_total(adata_new)
                sc.pp.log1p(adata_new)
            else:
                # print that data is raw counts
                print("Expression matrix contains raw counts. Proceeding with HVG selection.")
            # skip hvg selection if already n_genes is less than n_top_genes
            if adata_new.X.shape[1] <= n_top_genes:
                adata_new.var['highly_variable'] = np.array([True]*adata_new.X.shape[1])
                print("Number of genes less than or equal to n_top_genes. Skipping HVG selection.")
            else:
                sc.pp.highly_variable_genes(adata_new, n_top_genes=n_top_genes)

            #save hvgs
            #save hvgs
            hvg = adata.var['highly_variable']
            hvg_bools.append(hvg)
        
        #find union of hvgs
        hvg_selected = hvg_bools[0]
        for i in range(1, len(hvg_bools)):
            print(sum(hvg_selected), sum(hvg_bools[i]))
            hvg_selected = hvg_selected | hvg_bools[i]

        print("Number of HVGs: ", hvg_selected.sum())

    #filter expression matrices
    filtered_exp_mtxs = []
    for adata in adata_matrices:
        if not np.all(np.mod(adata.X.toarray(), 1) == 0):
            print("Warning: Expression matrix does not contain raw counts.")
            # Preprocess the data and subset
            sc.pp.normalize_total(adata)
            sc.pp.log1p(adata)
        else:
            print("Expression matrix contains raw counts. Proceeding with subsetting.")
        filtered_exp_mtxs.append(adata[:, hvg_selected].X)
    return filtered_exp_mtxs

hest1k_tenx115 = sc.read_h5ad('/lustre/scratch126/cellgen/lotfollahi/dj16/data/public_xenium/hest_1k_SKCM/TENX115/spatial_data_processed.h5')
hest1k_tenx117 = sc.read_h5ad('/lustre/scratch126/cellgen/lotfollahi/dj16/data/public_xenium/hest_1k_SKCM/TENX117/spatial_data_processed.h5')

filtered_mtx = hvg_selection_and_pooling([hest1k_tenx115, hest1k_tenx117])

for i in range(len(filtered_mtx)):
    # create folder if it does not exist
    filtered_mtx[i] = filtered_mtx[i].toarray()
    
    if not os.path.exists("/lustre/scratch126/cellgen/lotfollahi/kl11/spatial_alignment/data/GSE240429_data/data/filtered_expression_matrices/" + str(i+1)):
        os.makedirs("/lustre/scratch126/cellgen/lotfollahi/kl11/spatial_alignment/data/GSE240429_data/data/filtered_expression_matrices/" + str(i+1))
    np.save("/lustre/scratch126/cellgen/lotfollahi/kl11/spatial_alignment/data/GSE240429_data/data/filtered_expression_matrices/" + str(i+1) +"/hvg_matrix.npy", filtered_mtx[i].T)

(36601, 2378)
(36601, 2349)
(36601, 2277)
(36601, 2265)
1000 1000
1888 1000
2703 1000
Number of HVGs:  3467


In [ ]:
# read HEST1k

# print shapes
print("HEST1k TENX115 shape:", hest1k_tenx115.X.shape)
print("HEST1k TENX117 shape:", hest1k_tenx117.X.shape)
print("HEST1k TENX115 X:", hest1k_tenx115.X[:5, :5])
print("HEST1k TENX117 X:", hest1k_tenx117.X[:5, :5])

In [ ]:
################################
#! batch correct using harmony
#! Other batch correction methods can be used here in place of harmony. Furthermore, model can be trained using the hvg matrix and achieve comparable results if the datasets used are similar enough

d = np.load("/lustre/scratch126/cellgen/lotfollahi/kl11/spatial_alignment/data/GSE240429_data/data/filtered_expression_matrices/1/hvg_matrix.npy")
print(d.shape)

d2 = np.load("/lustre/scratch126/cellgen/lotfollahi/kl11/spatial_alignment/data/GSE240429_data/data/filtered_expression_matrices/2/hvg_matrix.npy")
print(d2.shape)


d = np.concatenate((d.T, d2.T), axis = 0)  

data_sizes = [105178, 87450]
batch_labels = np.concatenate((np.zeros(105178), np.ones(87450)))
batch_labels = batch_labels.astype(str)
df = pd.DataFrame(batch_labels, columns=["dataset"])

# # Run the Harmony integration algorithm
harmony = hm.run_harmony(d, meta_data=df, vars_use=["dataset"])
harmony_corrected = harmony.Z_corr.T

#split back into datasets
d1 = harmony_corrected[:data_sizes[0]]
d2 = harmony_corrected[data_sizes[0]:data_sizes[0]+data_sizes[1]]

print(d1.shape, d2.shape)

#save
np.save("/lustre/scratch126/cellgen/lotfollahi/kl11/spatial_alignment/data/GSE240429_data/data/filtered_expression_matrices/1/harmony_matrix.npy", d1.T)
np.save("/lustre/scratch126/cellgen/lotfollahi/kl11/spatial_alignment/data/GSE240429_data/data/filtered_expression_matrices/2/harmony_matrix.npy", d2.T)


In [ ]:
# can you plot d1 and d2 as pca to assess batch correction quality
# --- IGNORE ---
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
pca = PCA(n_components=2)
d1_pca = pca.fit_transform(d1)
d2_pca = pca.fit_transform(d2)
# --- IGNORE ---
plt.figure(figsize=(8,6))
plt.scatter(d1_pca[:,0], d1_pca[:,1], c='r', label='Dataset 1', alpha=0.5)
plt.scatter(d2_pca[:,0], d2_pca[:,1], c='b', label='Dataset 2', alpha=0.5)
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('PCA of Batch Corrected Data')
plt.legend()
# save figure
plt.savefig('/lustre/scratch126/cellgen/lotfollahi/kl11/spatial_alignment/data/batch_corrected_pca.png')